# Confusion Matrix & Classification Metrics (by hand, then verified)

When a model answers a **yes/no** question, "how good is it?" has more than one honest answer. This notebook builds the vocabulary for that from the ground up.

1. Train a simple **binary classifier** (logistic regression) on the breast-cancer dataset.
2. **By hand** with plain NumPy: count the four confusion-matrix cells (TP, TN, FP, FN) and compute **accuracy, precision, recall, F1** from the formulas.
3. **Draw** the 2x2 confusion matrix ourselves with matplotlib (no `ConfusionMatrixDisplay`).
4. **Verify** every hand-computed number against scikit-learn with `np.isclose` assertions.

The point is to demystify the metrics: once you have the four counts, everything else is arithmetic.

*(scikit-learn only supplies the data and the trained model — the metrics we care about are computed by hand first.)*

In [ ]:
import numpy as np                     # array math + the by-hand counting/comparisons
import matplotlib.pyplot as plt        # drawing the confusion matrix ourselves

from sklearn.datasets import load_breast_cancer          # a real, built-in binary dataset (offline)
from sklearn.model_selection import train_test_split     # split into train/test
from sklearn.linear_model import LogisticRegression      # the simple classifier we'll train
from sklearn.metrics import (                             # the 'answer key' we check ourselves against
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report,
)

# Seed everything up front so the split, the model, and every number below is reproducible
# run-to-run. (logistic regression here is deterministic, but seeding is a good habit.)
SEED = 42
np.random.seed(SEED)

print("imports ok")

## 1. Data and a simple classifier

`load_breast_cancer` gives 569 tumors described by 30 numeric features, each labelled **0 = malignant** or **1 = benign**. We split off a test set the model never sees during training, then fit a plain `LogisticRegression`.

A quick convention that matters for every metric below: we treat class **1 (benign) as the "positive" class**. "Positive" is just the label we're counting hits and misses against — it is *not* a value judgement.

In [ ]:
# Load features X (shape: 569 x 30) and integer labels y (0 or 1).
data = load_breast_cancer()
X, y = data.data, data.target

# Hold out 25% for testing. stratify=y keeps the same 0/1 ratio in both splits so the
# test set is a fair miniature of the whole dataset. random_state makes the split repeatable.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Fit logistic regression. max_iter is bumped up so the solver fully converges on this
# (unscaled) data without printing a convergence warning.
clf = LogisticRegression(max_iter=10000, random_state=SEED)
clf.fit(X_train, y_train)

# Predict hard 0/1 labels on the untouched test set. These predictions vs. the true
# y_test labels are the ONLY two arrays every metric below is built from.
y_pred = clf.predict(X_test)

print(f"test samples: {len(y_test)}")
print(f"actual positives (benign=1):    {int(np.sum(y_test == 1))}")
print(f"actual negatives (malignant=0): {int(np.sum(y_test == 0))}")
print(f"first 10 predictions: {y_pred[:10]}")
print(f"first 10 actuals:     {y_test[:10]}")

## 2. The four cells, by hand

Every prediction lands in exactly one of four buckets, depending on what we *predicted* vs. what was *actually* true (with **positive = 1**):

| | Predicted 1 (positive) | Predicted 0 (negative) |
|---|---|---|
| **Actual 1 (positive)** | **TP** — true positive (correct hit) | **FN** — false negative (missed it) |
| **Actual 0 (negative)** | **FP** — false positive (false alarm) | **TN** — true negative (correct pass) |

- **TP**: said positive, was positive. 
- **TN**: said negative, was negative. 
- **FP**: said positive, was actually negative (a *false alarm*). 
- **FN**: said negative, but it was actually positive (a *miss*). 

Each count is just a boolean AND of two conditions, summed. `np.sum` on a boolean array counts the `True`s.

In [ ]:
# Two boolean masks per cell, combined with & (elementwise AND), then counted with np.sum.
# .astype(int) at the end just turns the numpy integer into a plain Python int for clean printing.
TP = int(np.sum((y_test == 1) & (y_pred == 1)))   # actual 1 AND predicted 1  -> correct hit
TN = int(np.sum((y_test == 0) & (y_pred == 0)))   # actual 0 AND predicted 0  -> correct pass
FP = int(np.sum((y_test == 0) & (y_pred == 1)))   # actual 0 BUT predicted 1  -> false alarm
FN = int(np.sum((y_test == 1) & (y_pred == 0)))   # actual 1 BUT predicted 0  -> missed it

total = TP + TN + FP + FN   # the four buckets are mutually exclusive and cover every test sample

print(f"TP (true  positive, hit):         {TP}")
print(f"TN (true  negative, correct pass):{TN}")
print(f"FP (false positive, false alarm): {FP}")
print(f"FN (false negative, miss):        {FN}")
print(f"total = TP+TN+FP+FN = {total}  (should equal test size {len(y_test)})")

# Sanity check: the four cells must add up to the number of test samples, since every
# prediction falls into exactly one bucket.
assert total == len(y_test), "the four cells must cover every test sample exactly once"
print("count sanity check passed")

## 3. The metrics, by hand

With the four counts in hand, each metric is one small formula.

**Accuracy** — fraction of *all* predictions that were correct:
$$\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Precision** — of everything we *called positive*, how much really was? (Punishes false alarms.)
$$\text{precision} = \frac{TP}{TP + FP}$$

**Recall** (a.k.a. sensitivity / true-positive rate) — of everything that *was actually positive*, how much did we catch? (Punishes misses.)
$$\text{recall} = \frac{TP}{TP + FN}$$

**F1 score** — the *harmonic mean* of precision and recall, a single number that is only high when **both** are high:
$$F_1 = \frac{2 \cdot P \cdot R}{P + R}$$

In [ ]:
# Translate each formula above directly into code. We divide by explicit denominators so the
# link between formula and code is obvious. (All denominators are > 0 here because the model
# predicts both classes and both classes are present; in general you'd guard against 0.)

accuracy_hand  = (TP + TN) / total          # (TP + TN) / (TP + TN + FP + FN)
precision_hand = TP / (TP + FP)             # of predicted-positives, fraction truly positive
recall_hand    = TP / (TP + FN)             # of actual-positives, fraction we caught
f1_hand        = 2 * precision_hand * recall_hand / (precision_hand + recall_hand)  # harmonic mean

# Print each hand-computed value (4 decimals is plenty to eyeball).
print(f"accuracy  (by hand): {accuracy_hand:.4f}")
print(f"precision (by hand): {precision_hand:.4f}")
print(f"recall    (by hand): {recall_hand:.4f}")
print(f"F1        (by hand): {f1_hand:.4f}")

## 4. Draw the confusion matrix by hand

A confusion matrix is just those four numbers arranged in a 2x2 grid. We lay them out with **rows = Actual** and **columns = Predicted** (the same orientation scikit-learn uses, with labels ordered `[0, 1]`):

$$\begin{bmatrix} TN & FP \\ FN & TP \end{bmatrix}$$

We render it with `imshow` and annotate every cell with its count and its TP/FP/FN/TN label — no `ConfusionMatrixDisplay` helper, so nothing is hidden.

In [ ]:
# Build the 2x2 grid ourselves. Row index = actual class, column index = predicted class,
# both ordered [0, 1]. So row 0 = actual negative, row 1 = actual positive.
#   [ [TN, FP],        row 0 (actual 0): predicted 0 -> TN, predicted 1 -> FP
#     [FN, TP] ]       row 1 (actual 1): predicted 0 -> FN, predicted 1 -> TP
cm_hand = np.array([[TN, FP],
                    [FN, TP]])

# A parallel grid of short labels so each cell is annotated with BOTH its count and its name.
labels = np.array([["TN", "FP"],
                   ["FN", "TP"]])

fig, ax = plt.subplots(figsize=(5.5, 5))

# imshow paints the grid: darker/brighter color encodes the magnitude of each count.
im = ax.imshow(cm_hand, cmap="Blues")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="count")

# Tick positions 0 and 1, labelled with both the class number and its meaning.
ax.set_xticks([0, 1]); ax.set_xticklabels(["Predicted 0\n(malignant)", "Predicted 1\n(benign)"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Actual 0\n(malignant)", "Actual 1\n(benign)"])
ax.set_xlabel("Predicted label"); ax.set_ylabel("Actual label")
ax.set_title("Confusion matrix (drawn by hand)")

# Annotate each of the 4 cells with 'LABEL\ncount', centered. Use white text on dark cells
# and black on light cells so it stays readable regardless of the color intensity.
thresh = cm_hand.max() / 2.0   # midpoint of the color scale: above it the cell is 'dark'
for i in range(2):             # i = row = actual class
    for j in range(2):         # j = col = predicted class
        color = "white" if cm_hand[i, j] > thresh else "black"
        ax.text(j, i, f"{labels[i, j]}\n{cm_hand[i, j]}",
                ha="center", va="center", color=color, fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Verify against scikit-learn

Now the checkpoint: scikit-learn computes the same things with battle-tested code. If our by-hand numbers are right they must match. We use `np.isclose` (not `==`) because floating-point division can differ in the last bit, and we `assert` so a mismatch would *stop* the notebook loudly rather than pass silently.

In [ ]:
# sklearn's confusion_matrix returns rows=actual, cols=predicted, labels ordered [0, 1] ->
# exactly the [[TN, FP], [FN, TP]] layout we built by hand. So it should equal cm_hand cell-for-cell.
cm_sklearn = confusion_matrix(y_test, y_pred)

# The scalar metrics. By default these use pos_label=1, matching our 'positive = benign = 1' choice.
accuracy_sk  = accuracy_score(y_test, y_pred)
precision_sk = precision_score(y_test, y_pred)
recall_sk    = recall_score(y_test, y_pred)
f1_sk        = f1_score(y_test, y_pred)

# --- Assertions: by-hand MUST equal sklearn. Any failure raises and halts the notebook. ---
assert np.array_equal(cm_hand, cm_sklearn),        "confusion matrix grid mismatch"
assert np.isclose(accuracy_hand,  accuracy_sk),    "accuracy mismatch"
assert np.isclose(precision_hand, precision_sk),   "precision mismatch"
assert np.isclose(recall_hand,    recall_sk),      "recall mismatch"
assert np.isclose(f1_hand,        f1_sk),          "F1 mismatch"
print("All assertions passed: by-hand == sklearn\n")

# Side-by-side comparison table so you can see the numbers line up.
print(f"{'metric':<11}{'by hand':>12}{'sklearn':>12}{'match':>8}")
print("-" * 43)
for name, hand, sk in [
    ("accuracy",  accuracy_hand,  accuracy_sk),
    ("precision", precision_hand, precision_sk),
    ("recall",    recall_hand,    recall_sk),
    ("F1",        f1_hand,        f1_sk),
]:
    print(f"{name:<11}{hand:>12.4f}{sk:>12.4f}{str(np.isclose(hand, sk)):>8}")

print("\nConfusion matrix (sklearn) [[TN, FP], [FN, TP]]:")
print(cm_sklearn)

In [ ]:
# classification_report bundles precision/recall/F1 per class plus support (the count of true
# samples in each class). 'macro avg' averages the two classes equally; 'weighted avg' weights
# by support. Our metrics above are the row for class 1 (benign).
print(classification_report(y_test, y_pred, target_names=["malignant (0)", "benign (1)"]))

## 6. What the metrics mean, and when to trust them

**Precision vs. recall — the trade-off.**
- **Precision** asks: *when the model says positive, how often is it right?* Low precision = too many false alarms (FP).
- **Recall** asks: *of the real positives, how many did we catch?* Low recall = too many misses (FN).

These usually pull against each other. A classifier turns a probability into a label using a **threshold** (default 0.5). Lower the threshold and it says "positive" more often: recall goes **up** (fewer misses) but precision goes **down** (more false alarms). Raise it and the reverse happens. You can't freely maximize both — you pick the balance that fits the cost of each error.

*Which error hurts more depends on the problem.* Screening for a tumor, a **miss (FN)** is dangerous, so you favor **recall**. For a spam filter, dumping a real email (**FP**) annoys users, so you favor **precision**. **F1** is the single-number compromise — high only when precision *and* recall are both high.

**When accuracy is misleading.**
Accuracy = fraction correct, which sounds like the obvious score — but it hides everything when classes are **imbalanced**. If 99% of emails are legitimate, a model that blindly predicts "not spam" every time scores **99% accuracy** while catching **zero** spam (recall = 0). The accuracy looks great; the model is useless. That is why, on skewed datasets, you lean on **precision, recall, F1, and the confusion matrix** rather than accuracy alone — they expose *which* kind of mistake the model is making, not just how often it is wrong.

In this notebook the classes are fairly balanced (~63% benign / ~37% malignant), so accuracy is reasonable here — but the confusion matrix still tells you the fuller story at a glance.